In [4]:
import os

source_folder = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions"

# List all folders in the source directory
folders = [name for name in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, name))]
# Exclude non-prediction folders
exclude_names = {
    "automation scripts",
    "Full Daily Reports",
    "Logs",
    "Logs_Backup",
}
folders = [name for name in folders if name not in exclude_names]

print("Folders in '{}':".format(source_folder))
for folder in folders:
    print(folder)


Folders in 'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions':
CDC
CMS
DHA
FDA
HHS
HRSA
IHS
NIH
OS
VA


In [5]:
import pandas as pd

# Initialize a list to collect data
folder_data = []
# To store DataFrames for all CSVs
csv_data = []

# Loop through each folder and collect file info
for folder in folders:
    folder_path = os.path.join(source_folder, folder)
    files = os.listdir(folder_path)
    for file in files:
        file_path = os.path.join(folder_path, file)
        if os.path.isfile(file_path):
            # Collect file information
            file_info = {
                'folder': folder,
                'file_name': file,
                'file_path': file_path,
                'size_bytes': os.path.getsize(file_path),
                'modified_time': os.path.getmtime(file_path),
            }
            folder_data.append(file_info)

            # If the file is a CSV, open and read it into DataFrame
            if file.lower().endswith('.csv'):
                try:
                    df_csv = pd.read_csv(file_path)
                    df_csv['source_folder'] = folder
                    df_csv['source_file'] = file
                    csv_data.append(df_csv)
                except Exception as e:
                    print(f"Could not load {file_path}: {e}")

# Create a DataFrame from the data
df = pd.DataFrame(folder_data)

# Optional: Convert modified_time to datetime
df['modified_time'] = pd.to_datetime(df['modified_time'], unit='s')

# Display the DataFrame with file info
display(df.head())

# If any CSV data found, concatenate and display a sample
if csv_data:
    all_csv = pd.concat(csv_data, ignore_index=True)
    display(all_csv.head())
else:
    print("No CSV files loaded.")

,folder,file_name,file_path,size_bytes,modified_time
0,CDC,CDC_output_20250616.csv,Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\...,30894,2025-06-16 14:44:58.238551855
1,CDC,CDC_output_20250617.csv,Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\...,31590,2025-06-17 11:39:17.248404503
2,CDC,CDC_output_20250618.csv,Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\...,31810,2025-06-18 11:48:56.978536606
3,CDC,CDC_output_20250623.csv,Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\...,36045,2025-06-23 18:55:09.090246201
4,CDC,CDC_output_20250624.csv,Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\...,36292,2025-06-24 11:24:19.583616257


,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,source_folder,source_file,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,CDC_output_20250616.csv,NaN,NaN,NaN,NaN,NaN
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,CDC_output_20250616.csv,NaN,NaN,NaN,NaN,NaN
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,CDC_output_20250616.csv,NaN,NaN,NaN,NaN,NaN
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,CDC_output_20250616.csv,NaN,NaN,NaN,NaN,NaN
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,CDC_output_20250616.csv,NaN,NaN,NaN,NaN,NaN


In [6]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Configuration for OpDiv observation files
OPDIV_BASE_PATH = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#OPDIV_BASE_PATH = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
OPDIV_DATE_FORMAT = "%Y%m%d"

# Cover prediction history + max horizon (45d). Predictions start ~2025-06-16.
OPDIV_EVAL_START = datetime(2025, 6, 16)

base_path = OPDIV_BASE_PATH


def get_file_paths(base_path, start_date, end_date=None):
    """Generate existing observation file paths from start_date through end_date (inclusive)."""
    end = end_date or datetime.now()
    dates_to_pull = []
    d = start_date
    while d.date() <= end.date():
        dates_to_pull.append(d.strftime(OPDIV_DATE_FORMAT))
        d += timedelta(days=1)

    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    existing_files = [fp for fp in file_paths if os.path.exists(fp)]

    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(
            f"Observation files: {len(existing_files):,} "
            f"({existing_files[0].split('_d')[-1]} -> {existing_files[-1].split('_d')[-1]})"
        )

    return existing_files


def load_observed_data(file_paths):
    """Load indicator/OpDiv/obs_date only (memory-safe)."""
    data_frames = []
    usecols = ["indicator", "obs_date", "OpDiv"]

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path, usecols=usecols)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")

    if not data_frames:
        return pd.DataFrame(columns=usecols)

    observed_data_df = pd.concat(data_frames, ignore_index=True)
    display(f"Loaded data from {len(data_frames)} files | raw rows: {len(observed_data_df):,}")
    return observed_data_df


file_paths = get_file_paths(base_path, start_date=OPDIV_EVAL_START)
observed_data_df = load_observed_data(file_paths)
display(observed_data_df.head())
print(
    f"Unique indicators: {observed_data_df['indicator'].nunique():,} | "
    f"OpDivs: {sorted(observed_data_df['OpDiv'].dropna().astype(str).str.strip().unique())}"
)


'Observation files: 403 (20250616.csv -> 20260723.csv)'

'Loaded data from 403 files | raw rows: 1,582,095'

,indicator,obs_date,OpDiv
0,101.89.174.236,2025-06-16,CMS
1,101.89.174.236,2025-06-16,FDA
2,101.89.174.236,2025-06-16,HRSA
3,103.120.176.224,2025-06-16,VA
4,103.125.146.77,2025-06-16,HHS


Unique indicators: 11,971 | OpDivs: ['CDC', 'CMS', 'DHA', 'FDA', 'HHS', 'HRSA', 'IHS', 'NIH', 'OS', 'VA']


In [7]:
observed_data_df[observed_data_df['indicator'] == '104.18.8.59']

,indicator,obs_date,OpDiv
1376720,104.18.8.59,2026-06-02,VA
1381043,104.18.8.59,2026-06-03,VA
1385390,104.18.8.59,2026-06-04,CDC
1385391,104.18.8.59,2026-06-04,NIH
1385392,104.18.8.59,2026-06-04,VA
...,...,...,...
1569298,104.18.8.59,2026-07-21,VA
1573691,104.18.8.59,2026-07-22,CDC
1573692,104.18.8.59,2026-07-22,CMS
1573693,104.18.8.59,2026-07-22,IHS


In [8]:
# Overlap check: predictions vs observation archive
# Predictions use 'Indicator' + source_folder; observations use 'indicator' + OpDiv
all_csv_indicators = set(all_csv["Indicator"].dropna().astype(str).str.strip())
observed_src_indicators = set(observed_data_df["indicator"].dropna().astype(str).str.strip())

common_indicators = all_csv_indicators.intersection(observed_src_indicators)
num_common_indicators = len(common_indicators)

print(f"Indicators in predictions: {len(all_csv_indicators):,}")
print(f"Indicators in observations: {len(observed_src_indicators):,}")
print(f"Overlap (indicator only): {num_common_indicators:,}")

# OpDiv-aware overlap (what scoring actually uses)
pred_pairs = (
    all_csv.assign(
        indicator=lambda d: d["Indicator"].astype(str).str.strip(),
        opdiv=lambda d: d["source_folder"].astype(str).str.strip(),
    )[["indicator", "opdiv"]]
    .drop_duplicates()
)
obs_pairs = (
    observed_data_df.assign(
        indicator=lambda d: d["indicator"].astype(str).str.strip(),
        opdiv=lambda d: d["OpDiv"].astype(str).str.strip(),
    )[["indicator", "opdiv"]]
    .drop_duplicates()
)
merged_pairs = pred_pairs.merge(obs_pairs, on=["indicator", "opdiv"], how="inner")
print(f"Overlap (indicator + OpDiv): {len(merged_pairs):,} pairs")


Indicators in predictions: 10,115
Indicators in observations: 11,971
Overlap (indicator only): 9,308
Overlap (indicator + OpDiv): 30,439 pairs


In [9]:
import numpy as np
import gc
from collections import defaultdict

# =============================================================================
# PREP (memory-safe): all_csv = predictions | observed_data_df = multi-event GT
# Hit for horizon H (OpDiv-matched):
#   ANY obs with pred_date < obs_date <= pred_date + H
# Maturity: only score when pred_date + H <= max GT obs date
# =============================================================================

HORIZONS = {
    1:  {"prob": "Probability: 1-Day",  "conf": "Confidence: 1-Day"},
    7:  {"prob": "Probability: 7-Day",  "conf": "Confidence: 7-Day"},
    14: {"prob": "Probability: 14-Day", "conf": "Confidence: 14-Day"},
    30: {"prob": "Probability: 30-Day", "conf": "Confidence: 30-Day"},
    45: {"prob": "ensemble_45d",        "conf": "Confidence: 45-Day"},
}
if "Probability: 45-Day" in all_csv.columns:
    HORIZONS[45]["prob"] = "Probability: 45-Day"

keep_cols = ["Indicator", "source_folder", "source_file"]
for cols in HORIZONS.values():
    if cols["prob"] in all_csv.columns:
        keep_cols.append(cols["prob"])
    if cols["conf"] in all_csv.columns:
        keep_cols.append(cols["conf"])
keep_cols = list(dict.fromkeys(keep_cols))

pred = all_csv.loc[:, keep_cols].copy()
print(f"Working columns ({len(keep_cols)}): {keep_cols}")
print(f"Raw prediction rows: {len(pred):,}")

pred["indicator"] = pred["Indicator"].astype(str).str.strip()
pred["opdiv"] = pred["source_folder"].astype(str).str.strip()
pred["pred_date"] = pd.to_datetime(
    pred["source_file"].astype(str).str.extract(r"_(\d{8})", expand=False),
    format="%Y%m%d",
    errors="coerce",
)
pred.drop(columns=["Indicator", "source_folder", "source_file"], inplace=True, errors="ignore")

valid = pred["pred_date"].notna() & pred["indicator"].ne("nan") & pred["indicator"].ne("")
pred = pred.loc[valid]
pred["pred_date"] = pred["pred_date"].dt.normalize()

# ---- Ground truth: ALL observation events (do NOT collapse to latest-only) ----
gt = observed_data_df.loc[:, ["indicator", "OpDiv", "obs_date"]].copy()
gt["indicator"] = gt["indicator"].astype(str).str.strip()
gt["opdiv"] = gt["OpDiv"].astype(str).str.strip()
gt["obs_date"] = pd.to_datetime(gt["obs_date"], utc=True, errors="coerce")
gt = gt.loc[gt["obs_date"].notna() & gt["indicator"].ne("nan") & gt["indicator"].ne("")]
if getattr(gt["obs_date"].dt, "tz", None) is not None:
    gt["obs_date"] = gt["obs_date"].dt.tz_convert(None)
gt["obs_date"] = gt["obs_date"].dt.normalize()
gt = gt.drop_duplicates(subset=["indicator", "opdiv", "obs_date"])
gt.drop(columns=["OpDiv"], inplace=True, errors="ignore")

max_obs_date = gt["obs_date"].max()
min_obs_date = gt["obs_date"].min()
print(
    f"GT events: {len(gt):,} | indicators: {gt['indicator'].nunique():,} | "
    f"OpDivs: {gt['opdiv'].nunique()} | {min_obs_date.date()} -> {max_obs_date.date()}"
)

# Keep predictions only for (indicator, OpDiv) pairs that appear in GT
gt_keys = pd.MultiIndex.from_frame(gt[["indicator", "opdiv"]].drop_duplicates())
pred_keys = pd.MultiIndex.from_frame(pred[["indicator", "opdiv"]])
n_before = len(pred)
pred = pred.loc[pred_keys.isin(gt_keys)].copy()
print(f"After GT (indicator, OpDiv) filter: {len(pred):,} (dropped {n_before - len(pred):,})")

pred = pred.drop_duplicates(subset=["indicator", "opdiv", "pred_date"], keep="last")
pred.reset_index(drop=True, inplace=True)
print(f"After dedupe: {len(pred):,}")
gc.collect()


def parse_pct(series):
    """Parse '74.93%' or 74.93 into probability in [0, 1]."""
    s = series.astype(str).str.strip().str.replace("%", "", regex=False)
    vals = pd.to_numeric(s, errors="coerce")
    if vals.dropna().gt(1).any():
        vals = vals / 100.0
    return vals.clip(0, 1).astype("float32")


for h, cols in HORIZONS.items():
    if cols["prob"] in pred.columns:
        pred[f"prob_{h}d"] = parse_pct(pred[cols["prob"]])
        pred.drop(columns=[cols["prob"]], inplace=True)
    else:
        pred[f"prob_{h}d"] = np.float32(np.nan)
    if cols["conf"] in pred.columns:
        pred[f"conf_{h}d"] = pred[cols["conf"]].astype(str)
        pred.drop(columns=[cols["conf"]], inplace=True)
    else:
        pred[f"conf_{h}d"] = "Unknown"

# Sorted obs-date arrays per (indicator, opdiv) for fast window checks
obs_lookup = defaultdict(list)
for ind, opd, od in zip(gt["indicator"].to_numpy(), gt["opdiv"].to_numpy(), gt["obs_date"].to_numpy()):
    obs_lookup[(ind, opd)].append(np.datetime64(od, "D"))
obs_lookup = {k: np.sort(np.asarray(v, dtype="datetime64[D]")) for k, v in obs_lookup.items()}


def next_obs_after(indicator, opdiv, pred_date):
    """First observation strictly after pred_date, or NaT."""
    dates = obs_lookup.get((indicator, opdiv))
    if dates is None or len(dates) == 0:
        return pd.NaT
    start = np.datetime64(pd.Timestamp(pred_date).normalize(), "D")
    lo = np.searchsorted(dates, start, side="right")
    if lo >= len(dates):
        return pd.NaT
    return pd.Timestamp(dates[lo])


pred["next_obs_date"] = [
    next_obs_after(i, o, d)
    for i, o, d in zip(pred["indicator"], pred["opdiv"], pred["pred_date"])
]
pred["days_until_obs"] = (pred["next_obs_date"] - pred["pred_date"]).dt.days

# Maturity cutoff = last day covered by observation archive (not wall-clock alone)
as_of = max_obs_date
print(f"Predictions kept: {len(pred):,} | indicators: {pred['indicator'].nunique():,}")
print(f"Pred date range: {pred['pred_date'].min().date()} -> {pred['pred_date'].max().date()}")
print(f"As-of / maturity cutoff (max GT obs date): {as_of.date()}")
print("Note: evaluation is conditional on (indicator, OpDiv) appearing in GT at least once.")
gc.collect()


Working columns (13): ['Indicator', 'source_folder', 'source_file', 'Probability: 1-Day', 'Confidence: 1-Day', 'Probability: 7-Day', 'Confidence: 7-Day', 'Probability: 14-Day', 'Confidence: 14-Day', 'Probability: 30-Day', 'Confidence: 30-Day', 'ensemble_45d', 'Confidence: 45-Day']
Raw prediction rows: 4,166,649
GT events: 1,582,095 | indicators: 11,971 | OpDivs: 10 | 2025-06-16 -> 2026-07-23
After GT (indicator, OpDiv) filter: 4,062,768 (dropped 103,881)
After dedupe: 4,062,768
Predictions kept: 4,062,768 | indicators: 9,299
Pred date range: 2025-06-16 -> 2026-07-23
As-of / maturity cutoff (max GT obs date): 2026-07-23
Note: evaluation is conditional on (indicator, OpDiv) appearing in GT at least once.


18

In [10]:
# =============================================================================
# Evaluation: window hit vs confidence call
#
# Hit = ANY OpDiv-matched observation in (pred_date, pred_date + H]
#
# Graded results (used in accuracy):
#   Highly likely  + HIT  -> SUCCESS
#   Highly likely  + MISS -> FAIL
#   Low confidence + MISS -> SUCCESS
#   Low confidence + HIT  -> FAIL
#
# Possibly active:
#   tracked as HIT / MISS only — excluded from overall accuracy
#
# Precision = Highly likely hit rate
# Accuracy  = success rate among Highly likely + Low confidence only
# =============================================================================

def conf_bucket(text):
    t = str(text).lower()
    if "highly likely" in t:
        return "Highly likely"
    if "possibly active" in t:
        return "Possibly active"
    if "low confidence" in t:
        return "Low confidence"
    return "Unknown"


def observed_in_window(indicator, opdiv, pred_date, horizon):
    """True if any obs date is in (pred_date, pred_date + horizon]."""
    dates = obs_lookup.get((indicator, opdiv))
    if dates is None or len(dates) == 0:
        return False
    start = np.datetime64(pd.Timestamp(pred_date).normalize(), "D")  # exclusive
    end = np.datetime64(pd.Timestamp(pred_date).normalize() + pd.Timedelta(days=horizon), "D")
    lo = np.searchsorted(dates, start, side="right")
    hi = np.searchsorted(dates, end, side="right")
    return bool(hi > lo)


def grade_success(conf, hit):
    """Return 1/0 for graded calls, NaN for Possibly active / Unknown."""
    if conf == "Highly likely":
        return float(hit)          # hit -> success, miss -> fail
    if conf == "Low confidence":
        return float(1 - hit)      # miss -> success, hit -> fail
    return np.nan


def grade_result(conf, hit):
    if conf == "Highly likely":
        return "SUCCESS" if hit == 1 else "FAIL"
    if conf == "Low confidence":
        return "SUCCESS" if hit == 0 else "FAIL"
    if conf == "Possibly active":
        return "HIT" if hit == 1 else "MISS"
    return "—"


rows = []
for horizon in HORIZONS:
    prob_col = f"prob_{horizon}d"
    conf_col = f"conf_{horizon}d"

    m = pred[
        pred[prob_col].notna()
        & (pred["pred_date"] + pd.Timedelta(days=horizon) <= as_of)
    ].copy()
    if m.empty:
        print(f"{horizon}d: no matured predictions")
        continue

    m["hit"] = [
        int(observed_in_window(i, o, d, horizon))
        for i, o, d in zip(m["indicator"], m["opdiv"], m["pred_date"])
    ]
    m["horizon"] = horizon
    m["conf_bucket"] = m[conf_col].map(conf_bucket)
    m["probability"] = m[prob_col]
    m["success"] = [
        grade_success(c, h) for c, h in zip(m["conf_bucket"], m["hit"])
    ]
    m["result"] = [
        grade_result(c, h) for c, h in zip(m["conf_bucket"], m["hit"])
    ]
    rows.append(m)
    print(f"{horizon}d: scored {len(m):,} matured predictions | window hit rate {m['hit'].mean()*100:.1f}%")

scored = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

if scored.empty:
    print("No matured predictions to score.")
else:
    # ---- Consistency checks (fail fast if grading drifts) ----
    high = scored["conf_bucket"] == "Highly likely"
    low = scored["conf_bucket"] == "Low confidence"
    poss = scored["conf_bucket"] == "Possibly active"

    assert ((scored.loc[high, "success"] == scored.loc[high, "hit"]).all()), "Highly likely success must equal hit"
    assert ((scored.loc[low, "success"] == (1 - scored.loc[low, "hit"])).all()), "Low confidence success must equal 1-hit"
    assert scored.loc[poss, "success"].isna().all(), "Possibly active must be excluded from success/accuracy"
    assert set(scored.loc[high & (scored["hit"] == 1), "result"].unique()) <= {"SUCCESS"}
    assert set(scored.loc[high & (scored["hit"] == 0), "result"].unique()) <= {"FAIL"}
    assert set(scored.loc[low & (scored["hit"] == 0), "result"].unique()) <= {"SUCCESS"}
    assert set(scored.loc[low & (scored["hit"] == 1), "result"].unique()) <= {"FAIL"}
    assert set(scored.loc[poss & (scored["hit"] == 1), "result"].unique()) <= {"HIT"}
    assert set(scored.loc[poss & (scored["hit"] == 0), "result"].unique()) <= {"MISS"}
    print("Grading consistency checks passed.")

    print(
        f"\nScoring set: {len(scored):,} rows | "
        f"{scored['indicator'].nunique():,} indicators | "
        f"GT coverage through {as_of.date()}"
    )

    def indicator_stats(g):
        high_g = g[g["conf_bucket"] == "Highly likely"]
        low_g = g[g["conf_bucket"] == "Low confidence"]
        poss_g = g[g["conf_bucket"] == "Possibly active"]
        graded = g[g["success"].notna()]  # High + Low only

        return pd.Series({
            "n_predictions": len(g),
            "n_highly_likely": len(high_g),
            "n_low_confidence": len(low_g),
            "n_possibly_active": len(poss_g),
            "window_hits": int(g["hit"].sum()),
            "precision_%": round(high_g["hit"].mean() * 100, 1) if len(high_g) else np.nan,
            "accuracy_%": round(graded["success"].mean() * 100, 1) if len(graded) else np.nan,
            "possibly_hit_%": round(poss_g["hit"].mean() * 100, 1) if len(poss_g) else np.nan,
        })

    indicator_perf = (
        scored.groupby("indicator", sort=True)
        .apply(indicator_stats)
        .reset_index()
    )
    if "indicator" in indicator_perf.columns and indicator_perf.columns.duplicated().any():
        indicator_perf = indicator_perf.loc[:, ~indicator_perf.columns.duplicated()]

    print("Indicator prediction performance")
    display(indicator_perf)

    overall_rows = []
    for horizon, g in scored.groupby("horizon"):
        high_g = g[g["conf_bucket"] == "Highly likely"]
        low_g = g[g["conf_bucket"] == "Low confidence"]
        poss_g = g[g["conf_bucket"] == "Possibly active"]
        graded = g[g["success"].notna()]

        overall_rows.append({
            "horizon_days": horizon,
            "n_predictions": len(g),
            "n_indicators": g["indicator"].nunique(),
            "precision_%": round(high_g["hit"].mean() * 100, 1) if len(high_g) else np.nan,
            "accuracy_%": round(graded["success"].mean() * 100, 1) if len(graded) else np.nan,
            "highly_likely_n": len(high_g),
            "highly_likely_success_%": round(high_g["hit"].mean() * 100, 1) if len(high_g) else np.nan,
            "low_confidence_n": len(low_g),
            "low_confidence_success_%": round((1 - low_g["hit"]).mean() * 100, 1) if len(low_g) else np.nan,
            "possibly_active_n": len(poss_g),
            "possibly_active_hit_%": round(poss_g["hit"].mean() * 100, 1) if len(poss_g) else np.nan,
        })

    overall_perf = pd.DataFrame(overall_rows).sort_values("horizon_days")

    print("\nOverall model performance (accuracy excludes Possibly active)")
    display(overall_perf)

    high_all = scored[scored["conf_bucket"] == "Highly likely"]
    graded_all = scored[scored["success"].notna()]
    print(
        f"\nAll horizons combined — "
        f"precision (Highly likely hit rate): {high_all['hit'].mean()*100:.1f}% "
        f"(n={len(high_all):,}) | "
        f"accuracy (High+Low only): {graded_all['success'].mean()*100:.1f}% "
        f"(n={len(graded_all):,})"
    )


1d: scored 2,792,517 matured predictions | window hit rate 34.1%
7d: scored 3,934,398 matured predictions | window hit rate 47.3%
14d: scored 3,861,284 matured predictions | window hit rate 51.7%
30d: scored 3,680,399 matured predictions | window hit rate 56.0%
45d: scored 3,073,617 matured predictions | window hit rate 56.1%
Grading consistency checks passed.

Scoring set: 17,342,215 rows | 9,270 indicators | GT coverage through 2026-07-23
Indicator prediction performance


C:\Users\jaskew\AppData\Local\Temp\ipykernel_4472\404171624.py:136: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(indicator_stats)


,indicator,n_predictions,n_highly_likely,n_low_confidence,n_possibly_active,window_hits,precision_%,accuracy_%,possibly_hit_%
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,417.0,54.0,350.0,13.0,0.0,0.0,86.6,0.0
1,1-you.njalla.no,2351.0,522.0,1356.0,473.0,877.0,59.2,70.4,47.6
2,1.14.125.25,3.0,0.0,3.0,0.0,0.0,NaN,100.0,NaN
3,1.192.18.4,2513.0,469.0,1718.0,326.0,562.0,49.5,82.0,53.1
4,1.20.169.90,463.0,70.0,355.0,38.0,13.0,17.1,86.1,0.0
...,...,...,...,...,...,...,...,...,...
9265,yotpo-static.com,338.0,16.0,267.0,55.0,0.0,0.0,94.3,0.0
9266,yourmissourijudges.org,6.0,0.0,6.0,0.0,0.0,NaN,100.0,NaN
9267,yourpensionmeeting.com,2140.0,966.0,843.0,331.0,964.0,87.2,91.7,28.7
9268,yst.org,261.0,73.0,162.0,26.0,32.0,37.0,79.1,7.7



Overall model performance (accuracy excludes Possibly active)


,horizon_days,n_predictions,n_indicators,precision_%,accuracy_%,highly_likely_n,highly_likely_success_%,low_confidence_n,low_confidence_success_%,possibly_active_n,possibly_active_hit_%
0,1,2792517,8231,NaN,91.0,0,NaN,1942971,91.0,849546,91.4
1,7,3934398,9100,91.1,90.6,1671296,91.1,1955955,90.2,307147,48.0
2,14,3861284,8942,86.4,88.1,1970156,86.4,1598594,90.0,292534,46.0
3,30,3680399,8457,79.7,83.4,2334815,79.7,1096128,91.3,249456,41.8
4,45,3073617,7105,78.1,82.0,1996681,78.1,719125,92.8,357811,31.6



All horizons combined — precision (Highly likely hit rate): 83.4% (n=7,972,948) | accuracy (High+Low only): 86.9% (n=15,285,721)


In [11]:
# =============================================================================
# Clean scored view (uses result already graded in cell 6)
# =============================================================================

if "scored" not in dir() or scored.empty:
    print("No scored rows — run cells 5 and 6 first.")
else:
    scored_obs = scored.copy()
    print(
        f"{len(scored_obs):,} scored rows | "
        f"{scored_obs['indicator'].nunique():,} indicators | "
        f"GT through {as_of.date()}"
    )

    scored_clean = pd.DataFrame({
        "OpDiv": scored_obs["opdiv"],
        "Indicator": scored_obs["indicator"],
        "Pred Date": scored_obs["pred_date"].dt.strftime("%Y-%m-%d"),
        "Next Obs": scored_obs["next_obs_date"].dt.strftime("%Y-%m-%d"),
        "Days to Next Obs": scored_obs["days_until_obs"],
        "Horizon": scored_obs["horizon"].astype(str) + "d",
        "Probability %": (scored_obs["probability"] * 100).round(1),
        "Confidence": scored_obs["conf_bucket"],
        "Hit": scored_obs["hit"].map({1: "Yes", 0: "No"}),
        "Result": scored_obs["result"],
    })

    scored_clean = scored_clean.sort_values(
        ["Horizon", "Confidence", "OpDiv", "Indicator", "Pred Date"]
    ).reset_index(drop=True)

    n = len(scored_clean)
    n_hit = int((scored_obs["hit"] == 1).sum())
    graded = scored_obs[scored_obs["success"].notna()]
    high = scored_obs[scored_obs["conf_bucket"] == "Highly likely"]

    print(
        f"Window hits: {n_hit:,} ({n_hit/n*100:.1f}%)  |  "
        f"Precision (Highly likely): "
        f"{(high['hit'].mean()*100 if len(high) else float('nan')):.1f}%  |  "
        f"Accuracy (High+Low only): "
        f"{(graded['success'].mean()*100 if len(graded) else float('nan')):.1f}%"
    )
    print(
        "Result rules:\n"
        "  Highly likely  + hit  = SUCCESS | Highly likely  + miss = FAIL\n"
        "  Low confidence + miss = SUCCESS | Low confidence + hit  = FAIL\n"
        "  Possibly active       = HIT/MISS only (excluded from accuracy)\n"
        "Hit = any observation for that OpDiv in (pred_date, pred_date + horizon]\n"
    )

    display(scored_clean)

    overall_rows = []
    for horizon, g in scored_obs.groupby("horizon"):
        high_g = g[g["conf_bucket"] == "Highly likely"]
        low_g = g[g["conf_bucket"] == "Low confidence"]
        poss_g = g[g["conf_bucket"] == "Possibly active"]
        graded_g = g[g["success"].notna()]
        overall_rows.append({
            "horizon_days": horizon,
            "n_predictions": len(g),
            "n_indicators": g["indicator"].nunique(),
            "precision_%": round(high_g["hit"].mean() * 100, 1) if len(high_g) else np.nan,
            "accuracy_%": round(graded_g["success"].mean() * 100, 1) if len(graded_g) else np.nan,
            "highly_likely_n": len(high_g),
            "highly_likely_success_%": round(high_g["hit"].mean() * 100, 1) if len(high_g) else np.nan,
            "low_confidence_n": len(low_g),
            "low_confidence_success_%": round((1 - low_g["hit"]).mean() * 100, 1) if len(low_g) else np.nan,
            "possibly_active_n": len(poss_g),
            "possibly_active_hit_%": round(poss_g["hit"].mean() * 100, 1) if len(poss_g) else np.nan,
        })
    overall_perf = pd.DataFrame(overall_rows).sort_values("horizon_days")
    print("\nOverall model performance (accuracy excludes Possibly active)")
    display(overall_perf)


17,342,215 scored rows | 9,270 indicators | GT through 2026-07-23
Window hits: 8,596,312 (49.6%)  |  Precision (Highly likely): 83.4%  |  Accuracy (High+Low only): 86.9%
Result rules:
  Highly likely  + hit  = SUCCESS | Highly likely  + miss = FAIL
  Low confidence + miss = SUCCESS | Low confidence + hit  = FAIL
  Possibly active       = HIT/MISS only (excluded from accuracy)
Hit = any observation for that OpDiv in (pred_date, pred_date + horizon]



,OpDiv,Indicator,Pred Date,Next Obs,Days to Next Obs,Horizon,Probability %,Confidence,Hit,Result
0,CDC,102.129.153.158,2025-08-07,NaN,NaN,14d,73.199997,Highly likely,No,FAIL
1,CDC,102.129.153.158,2025-08-08,NaN,NaN,14d,73.199997,Highly likely,No,FAIL
2,CDC,102.129.153.158,2025-08-09,NaN,NaN,14d,73.199997,Highly likely,No,FAIL
3,CDC,102.129.153.158,2025-08-10,NaN,NaN,14d,73.199997,Highly likely,No,FAIL
4,CDC,102.129.153.158,2025-08-11,NaN,NaN,14d,66.900002,Highly likely,No,FAIL
...,...,...,...,...,...,...,...,...,...,...
17342210,VA,www.deepseek.com.cdn.cloudflare.net,2025-07-28,2025-07-29,1.0,7d,100.000000,Possibly active,Yes,HIT
17342211,VA,www.deepseek.com.cdn.cloudflare.net,2025-08-13,2025-08-14,1.0,7d,90.800003,Possibly active,Yes,HIT
17342212,VA,www.sthda.com,2025-08-11,NaN,NaN,7d,92.699997,Possibly active,No,MISS
17342213,VA,yourpensionmeeting.com,2025-07-16,2025-07-17,1.0,7d,59.400002,Possibly active,Yes,HIT



Overall model performance (accuracy excludes Possibly active)


,horizon_days,n_predictions,n_indicators,precision_%,accuracy_%,highly_likely_n,highly_likely_success_%,low_confidence_n,low_confidence_success_%,possibly_active_n,possibly_active_hit_%
0,1,2792517,8231,NaN,91.0,0,NaN,1942971,91.0,849546,91.4
1,7,3934398,9100,91.1,90.6,1671296,91.1,1955955,90.2,307147,48.0
2,14,3861284,8942,86.4,88.1,1970156,86.4,1598594,90.0,292534,46.0
3,30,3680399,8457,79.7,83.4,2334815,79.7,1096128,91.3,249456,41.8
4,45,3073617,7105,78.1,82.0,1996681,78.1,719125,92.8,357811,31.6


In [12]:
# =============================================================================
# Prob band 0.6–0.7: hit rate by frequency gate (freq >= 2 vs freq < 2)
#
# From classify_window (V4): Highly likely requires prob >= 0.6 AND freq >= 2.
# So within prob ≈ 0.6–0.7:
#   Highly likely              => freq >= 2
#   Possibly active / Low conf => freq < 2
# =============================================================================

PROB_LO, PROB_HI = 0.6, 0.7

if "scored" not in dir() or scored.empty:
    print("No scored rows — run cells 5 and 6 first.")
else:
    band = scored[
        scored["probability"].between(PROB_LO, PROB_HI, inclusive="both")
    ].copy()

    band["freq_gate"] = np.where(
        band["conf_bucket"] == "Highly likely",
        "freq >= 2 (Highly likely)",
        np.where(
            band["conf_bucket"].isin(["Possibly active", "Low confidence"]),
            "freq < 2 (Possibly / Low)",
            "Other",
        ),
    )
    band = band[band["freq_gate"] != "Other"]

    print(
        f"Probability band [{PROB_LO}, {PROB_HI}] | "
        f"{len(band):,} matured rows "
        f"({band['indicator'].nunique():,} indicators)\n"
    )

    rows = []
    for horizon, g in band.groupby("horizon"):
        for gate, gg in g.groupby("freq_gate"):
            rows.append({
                "horizon_days": horizon,
                "group": gate,
                "n": len(gg),
                "hit_rate_%": round(gg["hit"].mean() * 100, 1),
                "hits": int(gg["hit"].sum()),
            })

    # Pooled across horizons (multi-counts; secondary)
    for gate, gg in band.groupby("freq_gate"):
        rows.append({
            "horizon_days": "all (pooled)",
            "group": gate,
            "n": len(gg),
            "hit_rate_%": round(gg["hit"].mean() * 100, 1),
            "hits": int(gg["hit"].sum()),
        })

    freq_gate_perf = pd.DataFrame(rows)
    # Side-by-side pivot for readability
    pivot = (
        freq_gate_perf.pivot(index="horizon_days", columns="group", values=["n", "hit_rate_%"])
        .sort_index(key=lambda s: s.map(lambda x: 999 if isinstance(x, str) else x))
    )

    print("Hit rates in prob 0.6–0.7 by frequency gate")
    display(freq_gate_perf.sort_values(["horizon_days", "group"]))
    print("\nSide-by-side")
    display(pivot)

    # Clear one-liner comparison per horizon
    print("\nComparison (prob 0.6–0.7):")
    for horizon in sorted([h for h in band["horizon"].unique()]):
        g = band[band["horizon"] == horizon]
        hi = g[g["freq_gate"] == "freq >= 2 (Highly likely)"]
        lo = g[g["freq_gate"] == "freq < 2 (Possibly / Low)"]
        hi_rate = hi["hit"].mean() * 100 if len(hi) else float("nan")
        lo_rate = lo["hit"].mean() * 100 if len(lo) else float("nan")
        print(
            f"  {horizon:>2}d  "
            f"freq>=2 (Highly likely): {hi_rate:5.1f}% (n={len(hi):,})  |  "
            f"freq<2 (Poss/Low): {lo_rate:5.1f}% (n={len(lo):,})"
        )


Probability band [0.6, 0.7] | 526,527 matured rows (8,348 indicators)

Hit rates in prob 0.6–0.7 by frequency gate


,horizon_days,group,n,hit_rate_%,hits
0,1,freq < 2 (Possibly / Low),135843,80.1,108799
1,7,freq < 2 (Possibly / Low),76277,40.5,30863
2,7,freq >= 2 (Highly likely),24157,47.2,11402
3,14,freq < 2 (Possibly / Low),91514,38.8,35520
4,14,freq >= 2 (Highly likely),17693,39.4,6969
5,30,freq < 2 (Possibly / Low),86535,37.9,32779
6,30,freq >= 2 (Highly likely),10652,19.0,2028
7,45,freq < 2 (Possibly / Low),17700,41.4,7327
8,45,freq >= 2 (Highly likely),66156,34.2,22606
9,all (pooled),freq < 2 (Possibly / Low),407869,52.8,215288



Side-by-side


n                            \
group        freq < 2 (Possibly / Low) freq >= 2 (Highly likely)   
horizon_days                                                       
1                             135843.0                       NaN   
7                              76277.0                   24157.0   
14                             91514.0                   17693.0   
30                             86535.0                   10652.0   
45                             17700.0                   66156.0   
all (pooled)                  407869.0                  118658.0   

                            hit_rate_%                            
group        freq < 2 (Possibly / Low) freq >= 2 (Highly likely)  
horizon_days                                                      
1                                 80.1                       NaN  
7                                 40.5                      47.2  
14                                38.8                      39.4  
30                                37.9                      19.0  
45                                41.4                      34.2  
all (pooled)                      52.8                      36.2


Comparison (prob 0.6–0.7):
   1d  freq>=2 (Highly likely):   nan% (n=0)  |  freq<2 (Poss/Low):  80.1% (n=135,843)
   7d  freq>=2 (Highly likely):  47.2% (n=24,157)  |  freq<2 (Poss/Low):  40.5% (n=76,277)
  14d  freq>=2 (Highly likely):  39.4% (n=17,693)  |  freq<2 (Poss/Low):  38.8% (n=91,514)
  30d  freq>=2 (Highly likely):  19.0% (n=10,652)  |  freq<2 (Poss/Low):  37.9% (n=86,535)
  45d  freq>=2 (Highly likely):  34.2% (n=66,156)  |  freq<2 (Poss/Low):  41.4% (n=17,700)
